# Using _EGObox_ optimizer _Egor_

_Last Update: 30/03/2026_

## Installation

In [1]:
# %pip install egobox

We import _egobox_ as _egx_ for short.

In [2]:
import numpy as np
import egobox as egx

## Example 1 : Continuous optimization

### Test functions

In [3]:
xspecs_xsinx = [[0., 25.]]
n_cstr_xsinx = 0

def xsinx(x: np.ndarray) -> np.ndarray:
    x = np.atleast_2d(x)
    y = (x - 3.5) * np.sin((x - 3.5) / (np.pi))
    return y

In [4]:
xspecs_g24 = [[0., 3.], [0., 4.]]
n_cstr_g24 = 2

# Objective
def G24(point):
    """
    Function g24
    1 global optimum y_opt = -5.5080 at x_opt =(2.3295, 3.1785)
    """
    p = np.atleast_2d(point)
    return - p[:, 0] - p[:, 1]

# Constraints < 0
def G24_c1(point):
    p = np.atleast_2d(point)
    return (- 2.0 * p[:, 0] ** 4.0
            + 8.0 * p[:, 0] ** 3.0 
            - 8.0 * p[:, 0] ** 2.0 
            + p[:, 1] - 2.0)

def G24_c2(point):
    p = np.atleast_2d(point)
    return (-4.0 * p[:, 0] ** 4.0
            + 32.0 * p[:, 0] ** 3.0
            - 88.0 * p[:, 0] ** 2.0
            + 96.0 * p[:, 0]
            + p[:, 1] - 36.0)

# Grouped evaluation
def g24(point):
    p = np.atleast_2d(point)
    return np.array([G24(p), G24_c1(p), G24_c2(p)]).T


### Utility function

Simple utility function used below to display Egor minimize output.

In [5]:
def print_output(optim):
    result = optim.result
    status = optim.status

    print(f"Optimization {status.info.fname}({result.x_opt})={result.y_opt}")
    print("Optimization history: ")
    print(f"x: {result.x_doe}")
    print(f"y: {result.y_doe}")
    print("Optimization run status: ")
    print(f"  minimize: {status.info.fname}, run={status.info.num}")
    print(f"  exit status: {status.exit}")
    print(f"  init_doe_size: {status.init_doe_size}")
    print(f"  best_iter: {status.best_iter}")
    print(f"  total_iters: {status.total_iters}")
    print(f"  elapsed_time: {status.elapsed_time}ms")

### Continuous optimization

In [6]:
# Unconstrained 1D function
egor = egx.Egor(xspecs_xsinx)   # see help(egor) for options
optim = egor.minimize(xsinx, max_iters=30, run_info=egx.RunInfo(fname="xsinx", num=13))
print_output(optim)

Optimization xsinx([18.935211])=[-15.12510324]
Optimization history: 
x: [[19.14491989]
 [21.93964017]
 [ 5.93064174]
 [ 4.71246418]
 [11.02275439]
 [17.73370218]
 [18.82161103]
 [18.93509814]
 [18.93516978]
 [18.93521923]
 [18.935211  ]
 [18.93522959]
 [18.93521838]
 [ 0.        ]
 [18.93524262]
 [25.        ]
 [18.93529467]
 [18.93520704]
 [18.93518477]
 [18.93521695]
 [18.93520693]
 [18.93523301]
 [18.93518039]
 [18.93520287]
 [ 1.61276024]
 [ 8.63431012]
 [18.93528344]
 [18.9352158 ]
 [18.93521456]
 [18.93521074]
 [18.93524156]
 [18.93519353]
 [18.93518073]
 [18.93526403]
 [18.93518033]]
y: [[-15.08832313]
 [ -7.41215245]
 [  1.69849551]
 [  0.45640734]
 [  5.11140606]
 [-13.99948651]
 [-15.11444524]
 [-15.12510323]
 [-15.12510323]
 [-15.12510324]
 [-15.12510324]
 [-15.12510324]
 [-15.12510324]
 [  3.14127616]
 [-15.12510324]
 [ 11.42919546]
 [-15.12510323]
 [-15.12510324]
 [-15.12510324]
 [-15.12510324]
 [-15.12510324]
 [-15.12510324]
 [-15.12510324]
 [-15.12510324]
 [  1.06674794

In [7]:
# Constrained 2D function
egor = egx.Egor(xspecs_g24, 
                n_doe=5, 
                cstr_specs=[egx.CstrSpec.leq(0.0, tol=1e-3)] * n_cstr_g24,
                infill_strategy=egx.InfillStrategy.WB2,
                target=-5.5,
                trego=False,
            )   

In [8]:
optim = egor.minimize(g24, max_iters=30, run_info=egx.RunInfo(fname="g24"))
print_output(optim)

Optimization g24([2.32941029 3.17200688])=[-5.50141717 -0.00558907 -0.00700268]
Optimization history: 
x: [[1.55188011e+00 3.48962156e-01]
 [2.15510999e+00 3.74507254e+00]
 [6.28128538e-01 3.03278401e+00]
 [2.70963153e+00 1.72299175e+00]
 [2.82169173e-03 1.28133426e+00]
 [3.00000000e+00 3.39713668e+00]
 [2.62336883e+00 4.00000000e+00]
 [2.05032663e+00 4.00000000e+00]
 [1.72873773e+00 3.28496598e+00]
 [2.45273077e+00 2.54891618e+00]
 [2.32941029e+00 3.17200688e+00]]
y: [[-1.90084227e+00 -2.61827900e+00 -2.20584730e+00]
 [-5.90018253e+00  1.52158752e+00 -6.47699557e-02]
 [-3.66091255e+00 -4.52309807e-01 -7.91328950e-02]
 [-4.43262328e+00 -7.67163543e+00  7.37248221e-01]
 [-1.28415596e+00 -7.18729251e-01 -3.44484833e+01]
 [-6.39713668e+00 -1.66028633e+01  3.39713668e+00]
 [-6.62336883e+00 -3.34858459e+00  2.50470488e+00]
 [-6.05032663e+00  1.97870529e+00  2.02364943e-02]
 [-5.01370371e+00  8.45154047e-01 -1.48026204e-01]
 [-5.00164694e+00 -1.91717865e+00  2.05945048e-02]
 [-5.50141717e+00

### Egor as a service: ask-and-tell interface

When the user needs to be in control of the optimization loop, `Egor` can be used as a service. 

For instance with the `xsinx` objective function, we can do:

In [9]:
xlimits = [[0.0, 25.0]]
egor = egx.Egor(xlimits)

# initial doe
x_doe = egx.lhs(xlimits, 3, seed=42)
y_doe = xsinx(x_doe)
for _ in range(10): # run for 10 iterations
    x = egor.suggest(x_doe, y_doe, seed=42)  # ask for best location
    x_doe = np.concatenate((x_doe, x))
    y_doe = np.concatenate((y_doe, xsinx(x))) 
result = egor.best_result(x_doe, y_doe)
print(f"Optimization f={result.y_opt} at {result.x_opt}")

Optimization f=[-15.12510324] at [18.93521353]


## Example 2 : Mixed-integer optimization

### Test function

In [10]:
xspecs_mixint_xsinx = [egx.XSpec(egx.XType.INT, [0, 25])]
n_cstr_mixint_xsinx = 0

def mixint_xsinx(x: np.ndarray) -> np.ndarray:
    x = np.atleast_2d(x)
    if (np.abs(np.linalg.norm(np.floor(x))-np.linalg.norm(x))< 1e-8):
        y = (x - 3.5) * np.sin((x - 3.5) / (np.pi))
    else:
        raise ValueError(f"Bad input: mixint_xsinx accepts integer only, got {x}")
    return y

### Mixed-integer optimization with _Egor_

In [11]:
egor = egx.Egor(xspecs_mixint_xsinx, 
                     n_doe=3, 
                     infill_strategy=egx.InfillStrategy.EI,
                     target=-15.12,
                    )  # see help(egor) for options
optim = egor.minimize(mixint_xsinx, max_iters=30)
print_output(optim)

Optimization objective_function([19.])=[-15.12161154]
Optimization history: 
x: [[ 4.]
 [23.]
 [10.]
 [22.]
 [21.]
 [20.]
 [19.]]
y: [[  0.07924194]
 [ -1.48334497]
 [  5.70983099]
 [ -7.10960014]
 [-11.44370682]
 [-14.15453288]
 [-15.12161154]]
Optimization run status: 
  minimize: objective_function, run=1
  exit status: ExitStatus.TARGET_COST_REACHED
  init_doe_size: 3
  best_iter: 4
  total_iters: 4
  elapsed_time: 0.008ms


## Example 3 : More mixed-integer optimization

In the following example we see we can have other special integer type cases, where a component of x can take one value out of a list of ordered values (ORD type) or being like an enum value (ENUM type). Those types differ by the processing related to the continuous relaxation made behind the scene:
* For INT type, resulting float is rounded to the closest int value,
* For ORD type, resulting float is cast to closest value among the given valid ones,
* For ENUM type, one hot encoding is performed to give the resulting value.  

### Test function

In [12]:
# Objective function which takes [FLOAT, ENUM1, ENUM2, ORD] as input
# Note that ENUM values are passed as indice value eg either 0, 1 or 2 for a 3-sized enum  
def mixobj(X):
    # float
    x1 = X[:, 0]
    #  ENUM 1
    c1 = X[:, 1]
    x2 = c1 == 0
    x3 = c1 == 1
    x4 = c1 == 2
    #  ENUM 2
    c2 = X[:, 2]
    x5 = c2 == 0
    x6 = c2 == 1
    # int
    i = X[:, 3]

    y = (x2 + 2 * x3 + 3 * x4) * x5 * x1 + (x2 + 2 * x3 + 3 * x4) * x6 * 0.95 * x1 + i
    return y.reshape(-1, 1)

### Mixed-integer optimization with _Egor_

In [13]:
xtypes = [
    egx.XSpec(egx.XType.FLOAT, [-5.0, 5.0]),
    egx.XSpec(egx.XType.ENUM, tags=["blue", "red", "green"]),
    egx.XSpec(egx.XType.ENUM, xlimits=[2]),
    egx.XSpec(egx.XType.ORD, [0, 2, 3]),
]
egor = egx.Egor(xtypes)
optim = egor.minimize(mixobj, max_iters=10, seed=42)
print_output(optim)

Optimization objective_function([-5.  2.  0.  0.])=[-15.]
Optimization history: 
x: [[ 1.62883164e+00  0.00000000e+00  0.00000000e+00  3.00000000e+00]
 [-3.36198576e+00  1.00000000e+00  1.00000000e+00  2.00000000e+00]
 [ 4.03379217e+00  1.00000000e+00  1.00000000e+00  0.00000000e+00]
 [ 5.36985740e-05  2.00000000e+00  1.00000000e+00  3.00000000e+00]
 [-8.66923347e-01  0.00000000e+00  0.00000000e+00  3.00000000e+00]
 [ 2.60951201e+00  2.00000000e+00  0.00000000e+00  2.00000000e+00]
 [-2.42178777e+00  0.00000000e+00  1.00000000e+00  0.00000000e+00]
 [-4.89280551e+00  2.00000000e+00  0.00000000e+00  2.00000000e+00]
 [-5.00000000e+00  2.00000000e+00  1.00000000e+00  0.00000000e+00]
 [-5.00000000e+00  2.00000000e+00  1.00000000e+00  3.00000000e+00]
 [-5.00000000e+00  2.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-5.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-5.00000000e+00  2.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-5.00000000e+00  2.00000000e+00  0.00000000

Note that `x_opt` result contains indices for corresponding optional tags list hence the second component should be read as 0="red", 1="green", 2="blue", while the third component was unamed 0 correspond to first enum value and 1 to the second one.

## Usage

In [14]:
help(egor)

Help on Egor in module egobox object:

class Egor(builtins.object)
 |  Egor(xspecs, gp_config=None, n_cstr=0, cstr_specs=None, infill_n_start=Ellipsis, n_doe=0, x_doe=None, y_doe=None, infill_strategy=Ellipsis, feasible_infill_strategy=Ellipsis, cstr_infill=False, cstr_strategy=Ellipsis, qei_config=None, infill_optimizer=Ellipsis, trego=None, coego_n_coop=0, target=None, failsafe_strategy=Ellipsis, seed=None, verbose=None)
 |
 |  Optimizer constructor
 |
 |  Parameters
 |  ----------
 |  xspecs : list of XSpec, list of [lower, upper] or array[nx, 2]
 |      Specifications of the nx components of the input x (eg. len(xspecs) == nx),
 |      with XSpec(xtype=FLOAT|INT|ORD|ENUM, xlimits=[<f(xtype)>] or tags=[strings]).
 |      Depending on the x type we get the following for xlimits:
 |
 |      * when FLOAT: xlimits is [float lower_bound, float upper_bound],
 |      * when INT: xlimits is [int lower_bound, int upper_bound],
 |      * when ORD: xlimits is [float_1, float_2, ..., float_n],


In [15]:
help(egx.GpConfig)

Help on class GpConfig in module egobox:

class GpConfig(builtins.object)
 |  GpConfig(regr_spec=Ellipsis, corr_spec=Ellipsis, kpls_dim=Ellipsis, n_clusters=Ellipsis, recombination=Ellipsis, theta_init=Ellipsis, theta_bounds=Ellipsis, theta_n_start=Ellipsis, theta_max_eval=Ellipsis)
 |
 |  GP configuration used by `Egor` and `GpMix`
 |
 |  Methods defined here:
 |
 |  __repr__(self, /)
 |      Return repr(self).
 |
 |  ----------------------------------------------------------------------
 |  Static methods defined here:
 |
 |  __new__(*args, **kwargs)
 |      Create and return a new object.  See help(type) for accurate signature.
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  corr_spec
 |      (CorrelationSpec flags, an int in [1, 15])
 |      Specification of correlation models used in mixture.
 |      Can be CorrelationSpec.SQUARED_EXPONENTIAL (1), CorrelationSpec.ABSOLUTE_EXPONENTIAL (2),
 |      CorrelationSp